# PKG customer metrics v2: one row per customer per month

**PNC Treasury Management · Data Science · Payment Knowledge Graph**

**Inputs**
- the edge tables from `pkg_edges_2024_backfill.ipynb`;
- the MDM customer table `dsihd01p_dsi.neo4j_customer`.

**Output** (database `bdahd01p_dlcdi1_cdi_tm`)

| Table | Grain | Purpose |
|---|---|---|
| `pkg_cust_dim` | one row per `mdm_id`: every customer in `neo4j_customer` **plus** every customer seen in the edges but missing from it | identity: power ids, name, party type, NAICS, TM flag, family size, `pwr_shared`, first and last active month |
| `pkg_cust_metrics_monthly` | **one row per customer per month** | identity columns first, then every metric |
| `pkg_cust_metrics_dictionary` | one row per column | definition, family, scope, direction |
| `pkg_cust_pair_month` | customer × direction × scope × counterparty, per month | intermediate; reusable |

**New in v2**
- **Four-state churn with a 12-month lookback, split internal / external.**
  - **new:** not seen in the previous 12 months.
  - **reactivated:** seen in the previous 12 months, but not last month.
  - **retained:** seen last month and this month.
  - **lost:** seen last month, absent this month.

  So a counterparty seen in month 1, absent in month 2 and back in month 3 is **lost** in month 2 and **reactivated** (not new) in month 3.
- **Recurring relationships** (the attrition v8 definition): *standing* = present in ≥ 4 of the previous 6 months; *broken* = standing but absent this month. This comes with a count and a dollar share.
- **The same churn and recurring terms for counterparty banks**, per direction. The inbound-bank block was the best feature family in attrition v8.
- **`n_fi_ach_*`:** distinct banks on ACH only, matching attrition's `fin_out_n` definition.
- **`avg_ticket_*`.**
- **`selfpay_*`:** dollars to or from the customer's own name at other banks. The match is the exact normalized name, or the first 8 characters for organisations only.
- **`history_months`:** how many months of history stand behind the churn columns. It's below 12 in 2024, because the data starts in 2024-01.

## Conventions
- **Column names:** `{metric}_{scope}_{dir}`.
  - **Scope:** `int` = another PNC customer, `ext` = an external counterparty, `self` = the same `mdm_id` on both sides, `all` = `int` + `ext`.
  - **Direction:** `out` = paid, `in` = received.
- **Entity level:** external counterparties are counted at `ent_l1`, internal ones by `mdm_id`.
- **Customer attributes are current-state:** the latest `load_dt` per `mdm_id` is applied to every month.
- **TM customer** = has a `customer_pwr_id`; `master_pwr_id` is the ultimate parent.
- **Empty values:** shares are NULL when the denominator is 0; amounts and counts are 0 when there's no activity.

**How to run.** The churn state is **sequential**: each month needs the previous month's state, so the build starts at `HISTORY_START` (2024-01). Run all cells; months already built are skipped. For a quick trial, use the first two or three months. The default is the full history.

In [ ]:
# ---- Config -------------------------------------------------------------------------------
import calendar, datetime as dt
def month_range(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-")); out = []
    while (y, m) <= (y2, m2):
        out.append(f"{y:04d}-{m:02d}"); y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out
_t = dt.date.today().replace(day=1) - dt.timedelta(days=1)
LAST_COMPLETE = f"{_t.year:04d}-{_t.month:02d}"

CUST_TABLE    = "dsihd01p_dsi.neo4j_customer"
TGT_DB        = "bdahd01p_dlcdi1_cdi_tm"
PREFIX        = "pkg_"

HISTORY_START = "2024-01"                               # first month of reliable data: churn state starts empty here
RUN_MONTHS    = month_range(HISTORY_START, LAST_COMPLETE)   # trial: month_range("2024-01", "2024-03")
FORCE          = False                                  # rebuild months already written (needs the previous month's state)
BUILD_CUST_DIM = True

LOOKBACK      = 12                                      # months of history for new vs reactivated (≤ 30)
REC_WINDOW, REC_MIN = 6, 4                              # standing = present in ≥ REC_MIN of the previous REC_WINDOW months
KEEP_STATES   = 2                                       # churn-state months kept on disk (older ones are pruned)

RAILS = ["ACH", "WIRE", "RTP_PRT", "RTP_P2P", "CHECK", "PCARD"]
RAIL_OF_DIGIT = {"1": "ACH", "2": "ACH", "3": "ACH", "4": "WIRE", "5": "RTP_PRT", "6": "RTP_P2P",
                 "7": "CHECK", "8": "DEBIT_CARD", "9": "PCARD"}
NAICS_PLACEHOLDERS = ["******", "-1", "0", "00", "000000", "999999", "UNKNOWN", "N/A", "NA", "NULL", "NONE"]
MIN_NAICS_CP = 5
LEGAL = r"\b(N A|INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|THE)\b"

FILES = {"cust_pair_month": 8, "cust_metrics_monthly": 6}
CODEC = "snappy"
SHUFFLE_PARTITIONS = 400

In [ ]:
# ---- Setup. Compat shims: Cloudera SPARK3 PySpark 3.3 predates NumPy 1.24/2.0 and pandas 2.0
import numpy as np, pandas as pd, warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):          # removed in pandas 2.0; PySpark 3.3 createDataFrame still calls it
    pd.DataFrame.iteritems = pd.DataFrame.items
    pd.Series.iteritems = pd.Series.items
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pyspark.*")

from pyspark.sql import SparkSession, Window, functions as F, types as Tp
from pyspark import StorageLevel
from IPython.display import display
from contextlib import contextmanager
import time

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
print("Spark", spark.version, "| months:", RUN_MONTHS[0], "→", RUN_MONTHS[-1], f"({len(RUN_MONTHS)})")
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250);    pd.set_option("display.max_colwidth", 120)
MD = StorageLevel.MEMORY_AND_DISK

def show(pdf, title, pct=(), money=()):
    print(f"\n■ {title}   [{len(pdf):,} rows]")
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                fmt[c] = "{:.2%}"
        elif c in money:                            fmt[c] = "${:,.0f}"
        elif pd.api.types.is_integer_dtype(pdf[c]): fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):   fmt[c] = "{:,.3f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:    sty = sty.hide(axis="index")
    except AttributeError: sty = sty.hide_index()
    display(sty)

def nz(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

def name_key(c):
    """Upper-case, letters/digits/spaces only, legal suffixes removed. Used for self-payment matching."""
    s = F.regexp_replace(F.upper(c), "[^A-Z0-9 ]", " ")
    s = F.trim(F.regexp_replace(F.regexp_replace(s, LEGAL, " "), r"\s+", " "))
    return F.when(F.length(s) >= 3, s)

TIMINGS = []
@contextmanager
def timed(stage, month=None):
    t0 = time.time(); yield
    s = time.time() - t0
    TIMINGS.append({"month": month, "stage": stage, "seconds": s})
    print(f"  ⏱ {month or '':<8} {stage}: {s/60:,.1f} min")

HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def prev_month(m):
    y, mo = map(int, m.split("-"))
    return f"{y - 1:04d}-12" if mo == 1 else f"{y:04d}-{mo - 1:02d}"
def months_between(a, b):
    ya, ma = map(int, a.split("-")); yb, mb = map(int, b.split("-"))
    return (yb - ya) * 12 + (mb - ma)
GB = 1e9

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {TGT_DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
KEYS_T = ["edge_monthly", "edge_daily", "cpty_dim", "cpty_entity_map",
          "cust_dim", "cust_pair_month", "cust_metrics_monthly", "cust_metrics_dictionary"]
T = {k: f"{TGT_DB}.{PREFIX}{k}" for k in KEYS_T}
P = {k: f"{DB_LOC}/{PREFIX}{k}" for k in KEYS_T}
STATE_DIR = f"{DB_LOC}/{PREFIX}cust_churn_state"          # working state, not a Hive table
state_path = lambda kind, m: f"{STATE_DIR}/kind={kind}/month={m}"
for k in ("edge_monthly", "edge_daily", "cpty_dim", "cpty_entity_map"):
    assert exists(P[k]), f"{T[k]} not found: run pkg_edges_2024_backfill.ipynb first"

def ensure_table(key, schema, partitioned):
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in schema.fields if f.name != "month")
    part = " PARTITIONED BY (month STRING)" if partitioned else ""
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {T[key]} ({cols}){part} STORED AS PARQUET LOCATION '{P[key]}'")

def add_partition(key, m):
    spark.sql(f"ALTER TABLE {T[key]} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{P[key]}/month={m}'")

def month_done(key, m):
    return exists(f"{P[key]}/month={m}/_SUCCESS")

def write_month(df, key, m, part_col, sort_cols):
    (df.repartition(FILES[key], part_col).sortWithinPartitions(*sort_cols)
       .write.mode("overwrite").option("compression", CODEC).parquet(f"{P[key]}/month={m}"))
    ensure_table(key, df.schema, True); add_partition(key, m)

def write_full(df, key):
    """Non-partitioned table: rewrite files, then drop + re-create so the schema always matches (metadata only)."""
    df.write.mode("overwrite").option("compression", CODEC).parquet(P[key])
    spark.sql(f"DROP TABLE IF EXISTS {T[key]}")
    ensure_table(key, df.schema, False); spark.catalog.refreshTable(T[key])

def size_of(path):
    if not exists(path): return 0, 0
    c = fs_for(path).getContentSummary(JPath(path))
    return c.getLength(), c.getFileCount()
EM = spark.read.parquet(P["edge_monthly"])
print("target:", DB_LOC)

## §1 Customer dimension: `pkg_cust_dim`

- **From `neo4j_customer`:** one row per `mdm_id`, the latest `load_dt` (on a tie, the row with a `customer_pwr_id` wins).
- **Added:** every `mdm_id` that appears as a customer in the edges but not in `neo4j_customer`, with `in_neo4j_customer = false`.
- **`pwr_shared`** flags the rows whose `customer_pwr_id` is also held by another `mdm_id`.

In [ ]:
CUST_COLS = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
             "customer_start_dt", "naics_cd", "naics_desc", "load_dt"]
_have = {c.lower() for c in spark.table(CUST_TABLE).columns}
assert not (set(CUST_COLS) - _have), f"{CUST_TABLE} missing {set(CUST_COLS) - _have}"

if BUILD_CUST_DIM or not exists(P["cust_dim"]):
    with timed("customer dimension"):
        raw = (spark.table(CUST_TABLE).select(*[nz(c).alias(c) for c in CUST_COLS])
                    .filter(F.col("mdm_id").isNotNull()).withColumn("party_type", F.upper("party_type")))
        w = Window.partitionBy("mdm_id").orderBy(F.col("load_dt").desc_nulls_last(),
                                                 F.col("customer_pwr_id").isNotNull().desc())
        latest = (raw.withColumn("n_load_versions", F.count("*").over(Window.partitionBy("mdm_id")))
                     .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn"))
        nc = F.upper(F.col("naics_cd"))
        latest = (latest
            .withColumn("naics_status", F.when(nc.isNull(), "missing")
                                         .when(nc.isin(NAICS_PLACEHOLDERS) | ~nc.rlike(r"^[0-9]{2}"), "placeholder")
                                         .otherwise("valid"))
            .withColumn("naics2", F.when(F.col("naics_status") == "valid", F.substring("naics_cd", 1, 2)))
            .withColumn("is_tm", F.col("customer_pwr_id").isNotNull()))
        fam = latest.filter(F.col("master_pwr_id").isNotNull()).groupBy("master_pwr_id").agg(F.count("*").alias("family_size"))
        shared = (latest.filter(F.col("customer_pwr_id").isNotNull()).groupBy("customer_pwr_id")
                        .agg(F.count("*").alias("_n")).filter("_n > 1").select("customer_pwr_id", F.lit(True).alias("pwr_shared")))
        latest = (latest.join(fam, "master_pwr_id", "left").join(shared, "customer_pwr_id", "left")
                        .withColumn("in_neo4j_customer", F.lit(True)))
        seen = (EM.filter(F.col("src_kind") == "CUST").select(F.col("src_id").alias("mdm_id"), "month")
                  .unionByName(EM.filter(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("mdm_id"), "month"))
                  .groupBy("mdm_id").agg(F.min("month").alias("first_edge_month"), F.max("month").alias("last_edge_month")))
        dim = (latest.join(seen, "mdm_id", "full")
                 .withColumn("in_neo4j_customer", F.coalesce("in_neo4j_customer", F.lit(False)))
                 .withColumn("is_tm", F.coalesce("is_tm", F.lit(False)))
                 .withColumn("pwr_shared", F.coalesce("pwr_shared", F.lit(False)))
                 .withColumn("naics_status", F.coalesce("naics_status", F.lit("missing")))
                 .select("mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
                         "naics_cd", "naics_desc", "naics2", "naics_status", "is_tm", "pwr_shared", "family_size",
                         "customer_start_dt", "load_dt", "n_load_versions", "in_neo4j_customer",
                         "first_edge_month", "last_edge_month"))
        write_full(dim, "cust_dim")

CD = spark.table(T["cust_dim"]).persist(MD)
q = CD.agg(F.count("*").alias("mdm_ids"),
           F.sum(F.col("in_neo4j_customer").cast("int")).alias("in_neo4j_customer"),
           F.sum((~F.col("in_neo4j_customer")).cast("int")).alias("edges_only_not_in_neo4j_customer"),
           F.sum(F.col("first_edge_month").isNotNull().cast("int")).alias("seen_in_edges"),
           F.sum(F.col("is_tm").cast("int")).alias("tm_customers"),
           F.sum(F.col("pwr_shared").cast("int")).alias("pwr_shared_rows"),
           F.countDistinct("master_pwr_id").alias("master_pwr_ids"),
           F.max("load_dt").alias("latest_load_dt")).toPandas()
show(q, T["cust_dim"])
show(CD.groupBy("in_neo4j_customer", "is_tm", "party_type", "naics_status").count()
       .orderBy("in_neo4j_customer", "is_tm", "party_type", "naics_status").toPandas(),
     "Customers by source, TM flag, party type and NAICS status")

## §2 Monthly build

**For each month:**
1. **Perspective rows:** each edge read from each customer side it touches.
2. **Counterparty attributes:** entity id (`ent_l1`), bank, business-name flag and self-payment flag for external counterparties; TM flag, party type and NAICS2 for internal ones.
3. **Pairs**, written to `pkg_cust_pair_month`.
4. **Churn state updates**, for counterparties and banks: a 12-bit presence mask plus the last monthly amount per relationship.
5. **Metrics.**
6. **Exact reconciliation** of dollars in and out against the edges.

In [ ]:
RAIL_MAP = F.create_map(*[F.lit(x) for kv in RAIL_OF_DIGIT.items() for x in kv])
S3, S2, D = ["int", "ext", "self"], ["int", "ext"], ["in", "out"]
DIM_SLIM = spark.table(T["cpty_dim"]).select(F.col("cpty_id").alias("d_id"), F.col("fi").alias("o_fi"),
                                             F.col("is_business").alias("o_biz"), name_key(F.col("cpty_name")).alias("o_key"))
XW = (spark.table(T["cpty_entity_map"]).filter(F.col("ent_l1") != F.col("cpty_id"))
           .select(F.col("cpty_id").alias("xw_from"), F.col("ent_l1").alias("xw_to")))
CATTR = CD.select(F.col("mdm_id").alias("c_id"), F.col("is_tm").alias("o_is_tm"),
                  F.col("party_type").alias("o_party_type"), F.col("naics2").alias("o_naics2"))
OWN = CD.select(F.col("mdm_id").alias("cust"), F.col("naics2").alias("own_naics2"),
                name_key(F.col("customer_name")).alias("own_key"), F.col("party_type").alias("own_party"))

def perspective(m):
    mon = EM.filter(F.col("month") == m)
    keep = ["category", "is_self", "amount", "n_txn"]
    out_ = mon.filter(F.col("src_kind") == "CUST").select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"),
              F.col("dst_id").alias("oid"), F.col("dst_kind").alias("okind"), *keep)
    in_  = mon.filter(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"),
              F.col("src_id").alias("oid"), F.col("src_kind").alias("okind"), *keep)
    p = (out_.unionByName(in_)
             .withColumn("scope", F.when(F.col("is_self"), "self").when(F.col("okind") == "CUST", "int").otherwise("ext"))
             .withColumn("rail", F.element_at(RAIL_MAP, F.substring("category", 1, 1))))
    ext = (p.filter(F.col("scope") == "ext")
            .join(XW, F.col("oid") == F.col("xw_from"), "left")
            .withColumn("other", F.coalesce("xw_to", "oid")).drop("xw_from", "xw_to")
            .join(DIM_SLIM, F.col("other") == F.col("d_id"), "left").drop("d_id")
            .join(OWN.select("cust", "own_key", "own_party"), "cust", "left")
            .withColumn("o_self", F.coalesce(
                (F.col("own_key") == F.col("o_key")) |
                ((F.col("own_party") == "O") & (F.length("own_key") >= 8) & (F.length("o_key") >= 8)
                 & (F.substring("own_key", 1, 8) == F.substring("o_key", 1, 8))), F.lit(False)))
            .drop("own_key", "own_party", "o_key"))
    intr = (p.filter(F.col("scope") != "ext").withColumn("other", F.col("oid"))
             .join(CATTR, F.col("other") == F.col("c_id"), "left").drop("c_id"))
    return ext.unionByName(intr, allowMissingColumns=True).drop("oid", "okind", "is_self")

def build_pairs(m, persp):
    pr = (persp.filter(F.col("other").isNotNull())
               .groupBy("cust", "dir", "scope", "other")
               .agg(F.sum("amount").cast("decimal(22,2)").alias("amount"), F.sum("n_txn").cast("long").alias("n_txn"),
                    F.first("o_fi", ignorenulls=True).alias("o_fi"),
                    F.max(F.col("o_biz").cast("int")).cast("boolean").alias("o_biz"),
                    F.max(F.col("o_self").cast("int")).cast("boolean").alias("o_self"),
                    F.max(F.col("o_is_tm").cast("int")).cast("boolean").alias("o_is_tm"),
                    F.first("o_party_type", ignorenulls=True).alias("o_party_type"),
                    F.first("o_naics2", ignorenulls=True).alias("o_naics2")))
    write_month(pr, "cust_pair_month", m, "cust", ["cust", "dir", "other"])

# ---- churn state: (cust, dir, scope, other) → mask (bit0 = present last month … bit L-1 = L months ago), last_amt
STATE_SCHEMA = "cust string, dir string, scope string, other string, mask int, last_amt double"
FULL_MASK, REC_MASK = (1 << LOOKBACK) - 1, (1 << REC_WINDOW) - 1

def load_state(kind, m):
    """State as of the end of month m-1. Empty at HISTORY_START; otherwise it must exist on disk."""
    pm = prev_month(m)
    if m == HISTORY_START:
        return spark.createDataFrame([], STATE_SCHEMA)
    if not exists(state_path(kind, pm)):
        raise RuntimeError(f"churn state for {pm} ({kind}) missing: build months sequentially from {HISTORY_START}, "
                           f"or rebuild from the earliest month whose previous state exists")
    return spark.read.parquet(state_path(kind, pm))

def step_state(kind, m, cur):
    """cur: (cust, dir, scope, other, a) for month m. Returns the joined frame with churn flags; writes the new state."""
    K = ["cust", "dir", "scope", "other"]
    j = (load_state(kind, m).join(cur.withColumn("_c", F.lit(1)), K, "full")
           .withColumn("h", F.coalesce("mask", F.lit(0)))
           .withColumn("c", F.col("_c").isNotNull())
           .withColumn("in_prev", F.expr("(h & 1) = 1"))
           .withColumn("seen", F.col("h") != 0)
           .withColumn("standing", F.expr(f"bit_count(h & {REC_MASK}) >= {REC_MIN}"))
           .persist(MD))
    new_state = (j.select(*K, F.expr(f"(shiftleft(h, 1) | cast(c as int)) & {FULL_MASK}").cast("int").alias("mask"),
                          F.when(F.col("c"), F.col("a")).otherwise(F.col("last_amt")).alias("last_amt"))
                  .filter("mask != 0"))
    new_state.write.mode("overwrite").parquet(state_path(kind, m))
    return j

def prune_states(m):
    keep = {m}
    x = m
    for _ in range(KEEP_STATES - 1):
        x = prev_month(x); keep.add(x)
    for kind in ("cpty", "fi"):
        base = f"{STATE_DIR}/kind={kind}"
        if not exists(base): continue
        fs = fs_for(base)
        for s in fs.listStatus(JPath(base)):
            mm = s.getPath().getName().split("=", 1)[-1]
            if mm < m and mm not in keep:
                fs.delete(s.getPath(), True)

In [ ]:
def metrics_for(m, persp, PR):
    amt = F.col("amount").cast("double")
    c = lambda s, d: (F.col("scope") == s) & (F.col("dir") == d)
    isd = lambda d: F.col("dir") == d
    cnt = lambda cond: F.sum(F.when(cond, 1).otherwise(0))

    # ---- flows, compositions, rails, self-pay, ACH banks (perspective rows) ------------------
    ag = []
    for s in S3:
        for d in D:
            ag += [F.sum(F.when(c(s, d), F.col("amount"))).alias(f"amt_{s}_{d}"),
                   F.sum(F.when(c(s, d), F.col("n_txn"))).alias(f"ntxn_{s}_{d}")]
    for d in D:
        ag += [F.sum(F.when(c("ext", d) & F.col("o_biz"), amt)).alias(f"_biz_{d}"),
               F.sum(F.when(c("ext", d) & F.col("o_self"), amt)).alias(f"_selfpay_{d}"),
               F.sum(F.when(c("int", d) & F.col("o_is_tm"), amt)).alias(f"_tm_{d}"),
               F.sum(F.when(c("int", d) & (F.col("o_party_type") == "O"), amt)).alias(f"_org_{d}")]
        ag += [F.sum(F.when(isd(d) & (F.col("scope") != "self") & (F.col("rail") == r), amt)).alias(f"_rail_{r}_{d}")
               for r in RAILS]
    flow = persp.groupBy("cust").agg(*ag)
    ach = (persp.filter((F.col("scope") == "ext") & (F.col("rail") == "ACH") & F.col("o_fi").isNotNull())
                .select("cust", "dir", "o_fi").distinct()
                .groupBy("cust").agg(*[cnt(isd(d)).alias(f"n_fi_ach_{d}") for d in D]))

    # ---- breadth, concentration, banks, self-pay banks (pairs, self excluded) ---------------------
    NS = PR.filter(F.col("scope") != "self").withColumn("a", amt)
    ag = [cnt(c(s, d)).alias(f"ncpty_{s}_{d}") for s in S2 for d in D]
    for d in D:
        ag += [F.sum(F.when(isd(d), F.col("a"))).alias(f"_tot_{d}"), F.max(F.when(isd(d), F.col("a"))).alias(f"_max_{d}"),
               F.sum(F.when(isd(d), F.col("a") * F.col("a"))).alias(f"_sq_{d}")]
    conc = NS.groupBy("cust").agg(*ag)
    fiagg = (NS.filter((F.col("scope") == "ext") & F.col("o_fi").isNotNull())
               .groupBy("cust", "dir", "o_fi").agg(F.sum("a").alias("a"), F.max(F.col("o_self").cast("int")).alias("s")))
    fi = fiagg.groupBy("cust").agg(*[x for d in D for x in (
             cnt(isd(d)).alias(f"n_fi_{d}"), F.max(F.when(isd(d), F.col("a"))).alias(f"_fimax_{d}"),
             F.sum(F.when(isd(d), F.col("a"))).alias(f"_fitot_{d}"),
             cnt(isd(d) & (F.col("s") == 1)).alias(f"selfpay_n_fi_{d}"))])

    # ---- sector mix of internal counterparties ----------------------------------------------------
    nx = NS.filter((F.col("scope") == "int") & F.col("o_naics2").isNotNull()).join(OWN.select("cust", "own_naics2"), "cust", "left")
    g = nx.groupBy("cust", "dir", "o_naics2").agg(F.sum("a").alias("a"), F.count("*").alias("n"))
    g = g.withColumn("p", F.col("a") / F.sum("a").over(Window.partitionBy("cust", "dir")))
    sect = g.groupBy("cust").agg(*[x for d in D for x in (
                cnt(isd(d)).alias(f"_nn_{d}"), F.sum(F.when(isd(d), F.col("n")).otherwise(0)).alias(f"_ncp_{d}"),
                F.max(F.when(isd(d), F.col("p"))).alias(f"_ntop_{d}"),
                F.sum(F.when(isd(d), -F.col("p") * F.log(F.col("p")))).alias(f"_nent_{d}"))])
    same = nx.filter(F.col("own_naics2").isNotNull()).groupBy("cust").agg(*[x for d in D for x in (
                F.sum(F.when(isd(d) & (F.col("o_naics2") == F.col("own_naics2")), F.col("a")).otherwise(0.0)).alias(f"_same_{d}"),
                F.sum(F.when(isd(d), F.col("a"))).alias(f"_valid_{d}"))])

    # ---- churn + recurring: counterparties -----------------------------------------------------------
    J = step_state("cpty", m, NS.select("cust", "dir", "scope", "other", "a"))
    ag = []
    for s in S2:
        for d in D:
            ag += [cnt(c(s, d) & F.col("c") & ~F.col("seen")).alias(f"new_cpty_{s}_{d}"),
                   cnt(c(s, d) & F.col("c") & F.col("seen") & ~F.col("in_prev")).alias(f"reactivated_cpty_{s}_{d}"),
                   cnt(c(s, d) & F.col("c") & F.col("in_prev")).alias(f"retained_cpty_{s}_{d}"),
                   cnt(c(s, d) & ~F.col("c") & F.col("in_prev")).alias(f"lost_cpty_{s}_{d}")]
    for d in D:
        ag += [cnt(isd(d) & F.col("in_prev")).alias(f"_prevn_{d}"),
               cnt(isd(d) & F.col("c") & F.col("in_prev")).alias(f"_retn_{d}"),
               F.sum(F.when(isd(d) & F.col("c") & ~F.col("seen"), F.col("a"))).alias(f"_newa_{d}"),
               F.sum(F.when(isd(d) & F.col("in_prev"), F.col("last_amt"))).alias(f"_preva_{d}"),
               F.sum(F.when(isd(d) & ~F.col("c") & F.col("in_prev"), F.col("last_amt"))).alias(f"_losta_{d}"),
               cnt(isd(d) & F.col("standing")).alias(f"rec_standing_{d}"),
               cnt(isd(d) & F.col("standing") & ~F.col("c")).alias(f"rec_broken_{d}"),
               F.sum(F.when(isd(d) & F.col("standing"), F.col("last_amt"))).alias(f"_stda_{d}"),
               F.sum(F.when(isd(d) & F.col("standing") & ~F.col("c"), F.col("last_amt"))).alias(f"_brka_{d}")]
    churn = J.groupBy("cust").agg(*ag)
    gone = J.filter(F.col("in_prev") & ~F.col("c")).select("cust")          # customers whose relationships all ended still get a row

    # ---- churn + recurring: banks (external) -----------------------------------------------------------
    cur_fi = fiagg.select("cust", "dir", F.lit("fi").alias("scope"), F.col("o_fi").alias("other"), "a")
    JF = step_state("fi", m, cur_fi)
    ag = []
    for d in D:
        ag += [cnt(isd(d) & F.col("c") & ~F.col("seen")).alias(f"new_fi_{d}"),
               cnt(isd(d) & F.col("c") & F.col("seen") & ~F.col("in_prev")).alias(f"reactivated_fi_{d}"),
               cnt(isd(d) & F.col("c") & F.col("in_prev")).alias(f"retained_fi_{d}"),
               cnt(isd(d) & ~F.col("c") & F.col("in_prev")).alias(f"lost_fi_{d}"),
               cnt(isd(d) & F.col("standing")).alias(f"_fistd_{d}"),
               cnt(isd(d) & F.col("standing") & ~F.col("c")).alias(f"fi_rec_broken_{d}")]
    fichurn = JF.groupBy("cust").agg(*ag)

    # ---- active days (daily edges) --------------------------------------------------------------------
    dd = spark.read.parquet(f"{P['edge_daily']}/month={m}").filter(~F.col("is_self"))
    act = (dd.filter(F.col("src_kind") == "CUST").select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"), "txn_date")
             .unionByName(dd.filter(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"), "txn_date"))
             .distinct().groupBy("cust").agg(*[cnt(isd(d)).alias(f"active_days_{d}") for d in D]))

    base = persp.select("cust").unionByName(gone).distinct()
    X = base
    for part in (flow, ach, conc, fi, sect, same, churn, fichurn, act):
        X = X.join(part, "cust", "left")

    # ---- derived columns -------------------------------------------------------------------------------
    z  = lambda col: F.coalesce(F.col(col).cast("double"), F.lit(0.0))
    sd = lambda num, den: F.when(den > 0, num / den)
    zero = lambda col, t="long": F.coalesce(F.col(col), F.lit(0)).cast(t)
    for s in S3:
        for d in D:
            X = X.withColumn(f"amt_{s}_{d}", zero(f"amt_{s}_{d}", "decimal(22,2)")).withColumn(f"ntxn_{s}_{d}", zero(f"ntxn_{s}_{d}"))
    for s in S2:
        for d in D:
            X = X.withColumn(f"ncpty_{s}_{d}", zero(f"ncpty_{s}_{d}"))
            for st in ("new", "reactivated", "retained", "lost"):
                X = X.withColumn(f"{st}_cpty_{s}_{d}", zero(f"{st}_cpty_{s}_{d}"))
    for d in D:
        X = (X.withColumn(f"amt_all_{d}", (F.col(f"amt_int_{d}") + F.col(f"amt_ext_{d}")).cast("decimal(22,2)"))
              .withColumn(f"ntxn_all_{d}", F.col(f"ntxn_int_{d}") + F.col(f"ntxn_ext_{d}"))
              .withColumn(f"ncpty_all_{d}", F.col(f"ncpty_int_{d}") + F.col(f"ncpty_ext_{d}")))
        A, E, I = z(f"amt_all_{d}"), z(f"amt_ext_{d}"), z(f"amt_int_{d}")
        for col in (f"active_days_{d}", f"n_fi_{d}", f"n_fi_ach_{d}", f"selfpay_n_fi_{d}", f"rec_standing_{d}", f"rec_broken_{d}",
                    f"new_fi_{d}", f"reactivated_fi_{d}", f"retained_fi_{d}", f"lost_fi_{d}", f"fi_rec_broken_{d}"):
            X = X.withColumn(col, zero(col))
        X = (X.withColumn(f"avg_ticket_{d}", sd(A, z(f"ntxn_all_{d}")))
              .withColumn(f"ext_share_{d}", sd(E, A))
              .withColumn(f"top1_share_{d}", sd(z(f"_max_{d}"), z(f"_tot_{d}")))
              .withColumn(f"hhi_{d}", sd(z(f"_sq_{d}"), z(f"_tot_{d}") * z(f"_tot_{d}")))
              .withColumn(f"top_fi_share_{d}", sd(z(f"_fimax_{d}"), z(f"_fitot_{d}")))
              .withColumn(f"fi_coverage_{d}", sd(z(f"_fitot_{d}"), E))
              .withColumn(f"ext_business_share_{d}", sd(z(f"_biz_{d}"), E))
              .withColumn(f"selfpay_amt_{d}", F.round(z(f"_selfpay_{d}"), 2).cast("decimal(22,2)"))
              .withColumn(f"selfpay_share_{d}", sd(z(f"_selfpay_{d}"), E))
              .withColumn(f"int_tm_share_{d}", sd(z(f"_tm_{d}"), I))
              .withColumn(f"int_org_share_{d}", sd(z(f"_org_{d}"), I)))
        guard = z(f"_ncp_{d}") >= MIN_NAICS_CP
        X = (X.withColumn(f"n_naics2_{d}", F.when(guard, F.col(f"_nn_{d}")).cast("long"))
              .withColumn(f"naics2_top_share_{d}", F.when(guard, F.col(f"_ntop_{d}")))
              .withColumn(f"naics2_entropy_{d}", F.when(guard, F.col(f"_nent_{d}")))
              .withColumn(f"same_naics2_share_{d}", sd(z(f"_same_{d}"), z(f"_valid_{d}"))))
        for r in RAILS:
            X = X.withColumn(f"share_{r}_{d}", sd(z(f"_rail_{r}_{d}"), A))
        X = X.withColumn(f"n_rails_{d}", sum((z(f"_rail_{r}_{d}") > 0).cast("int") for r in RAILS))
        X = (X.withColumn(f"retention_{d}", sd(z(f"_retn_{d}"), z(f"_prevn_{d}")))
              .withColumn(f"new_amt_share_{d}", sd(z(f"_newa_{d}"), A))
              .withColumn(f"lost_amt_share_{d}", sd(z(f"_losta_{d}"), z(f"_preva_{d}")))
              .withColumn(f"rec_broken_share_{d}", sd(z(f"rec_broken_{d}"), z(f"rec_standing_{d}")))
              .withColumn(f"rec_broken_amt_share_{d}", sd(z(f"_brka_{d}"), z(f"_stda_{d}")))
              .withColumn(f"fi_retention_{d}", sd(z(f"retained_fi_{d}"), z(f"retained_fi_{d}") + z(f"lost_fi_{d}")))
              .withColumn(f"fi_rec_broken_share_{d}", sd(z(f"fi_rec_broken_{d}"), z(f"_fistd_{d}"))))
    X = (X.withColumn("net_flow_all", (F.col("amt_all_in") - F.col("amt_all_out")).cast("decimal(22,2)"))
          .withColumn("net_flow_ext", (F.col("amt_ext_in") - F.col("amt_ext_out")).cast("decimal(22,2)"))
          .withColumn("passthrough_ratio", sd(F.least(z("amt_all_in"), z("amt_all_out")), F.greatest(z("amt_all_in"), z("amt_all_out"))))
          .withColumn("history_months", F.lit(min(LOOKBACK, months_between(HISTORY_START, m)))))

    ident = CD.select(F.col("mdm_id").alias("cust"), *[c_ for c_ in IDENTITY_COLS if c_ != "mdm_id"])
    X = X.join(ident, "cust", "left").withColumnRenamed("cust", "mdm_id") \
         .withColumn("in_neo4j_customer", F.coalesce("in_neo4j_customer", F.lit(False))) \
         .withColumn("is_tm", F.coalesce("is_tm", F.lit(False))) \
         .withColumn("pwr_shared", F.coalesce("pwr_shared", F.lit(False)))
    return X.select(*IDENTITY_COLS, *[c_ for c_, *_ in METRIC_SPEC]), J, JF

In [ ]:
# ---- Column order and data dictionary (single source of truth) ---------------------------------
IDENTITY_COLS = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "naics_cd", "naics_desc",
                 "naics2", "naics_status", "is_tm", "pwr_shared", "family_size", "customer_start_dt", "in_neo4j_customer"]
IDENTITY_DEF = {
    "mdm_id": "MDM party id (bank-wide customer key)",
    "customer_pwr_id": "TM customer power id; null = not a TM customer",
    "master_pwr_id": "ultimate parent power id of the customer's family",
    "customer_name": "customer name (latest load_dt)",
    "party_type": "P = person, O = organisation",
    "naics_cd": "NAICS code as recorded (populated for TM customers)",
    "naics_desc": "NAICS description",
    "naics2": "2-digit NAICS sector, only when naics_status = valid",
    "naics_status": "valid / placeholder (******, UNKNOWN, …) / missing",
    "is_tm": "true when the customer has a customer_pwr_id",
    "pwr_shared": "true when this customer_pwr_id is also held by another mdm_id (joins on power id are ambiguous)",
    "family_size": "number of customers sharing this master_pwr_id",
    "customer_start_dt": "customer start date",
    "in_neo4j_customer": "false when the mdm_id appears in payments but not in neo4j_customer (identity columns then null)",
    "month": "partition: activity month YYYY-MM"}
SCOPE_TXT = {"int": "other PNC customers", "ext": "external counterparties", "self": "the customer's own accounts (same mdm_id)",
             "all": "PNC customers + external counterparties (self-transfers excluded)"}
DIR_TXT = {"in": "received", "out": "paid"}
WITH_TXT = {"in": "from", "out": "to"}
M = []   # (column, family, scope, direction, definition)
for d in D:
    for s in ["int", "ext", "self", "all"]:
        M += [(f"amt_{s}_{d}", "flow", s, d, f"dollars {DIR_TXT[d]}, {SCOPE_TXT[s]}"),
              (f"ntxn_{s}_{d}", "flow", s, d, f"transactions {DIR_TXT[d]}, {SCOPE_TXT[s]}")]
    M += [(f"avg_ticket_{d}", "flow", "all", d, f"amt_all_{d} / ntxn_all_{d}: average payment size {DIR_TXT[d]}"),
          (f"active_days_{d}", "flow", "all", d, f"distinct posting dates with at least one transaction {DIR_TXT[d]}"),
          (f"ext_share_{d}", "flow", "all", d, f"share of dollars {DIR_TXT[d]} that involve external counterparties")]
    for s in ["int", "ext", "all"]:
        M.append((f"ncpty_{s}_{d}", "breadth", s, d, f"distinct counterparties {DIR_TXT[d]} {WITH_TXT[d]}, {SCOPE_TXT[s]} (external at entity level ent_l1)"))
    M += [(f"top1_share_{d}", "concentration", "all", d, f"share of dollars {DIR_TXT[d]} with the single largest counterparty"),
          (f"hhi_{d}", "concentration", "all", d, f"Herfindahl index of dollars {DIR_TXT[d]} across counterparties (1 = one counterparty)")]
    for s in S2:
        M += [(f"new_cpty_{s}_{d}", "churn", s, d, f"counterparties {DIR_TXT[d]} {WITH_TXT[d]} this month, not seen in the previous {LOOKBACK} months ({SCOPE_TXT[s]})"),
              (f"reactivated_cpty_{s}_{d}", "churn", s, d, f"counterparties {DIR_TXT[d]} {WITH_TXT[d]} this month, seen in the previous {LOOKBACK} months but not last month ({SCOPE_TXT[s]})"),
              (f"retained_cpty_{s}_{d}", "churn", s, d, f"counterparties {DIR_TXT[d]} {WITH_TXT[d]} both last month and this month ({SCOPE_TXT[s]})"),
              (f"lost_cpty_{s}_{d}", "churn", s, d, f"counterparties {DIR_TXT[d]} {WITH_TXT[d]} last month but not this month ({SCOPE_TXT[s]})")]
    M += [(f"retention_{d}", "churn", "all", d, f"retained / last month's counterparties, {DIR_TXT[d]}"),
          (f"new_amt_share_{d}", "churn", "all", d, f"share of this month's dollars {DIR_TXT[d]} that involve new counterparties"),
          (f"lost_amt_share_{d}", "churn", "all", d, f"share of last month's dollars {DIR_TXT[d]} that were with counterparties lost this month"),
          (f"rec_standing_{d}", "recurring", "all", d, f"standing relationships {DIR_TXT[d]}: present in ≥ {REC_MIN} of the previous {REC_WINDOW} months"),
          (f"rec_broken_{d}", "recurring", "all", d, f"standing relationships {DIR_TXT[d]} absent this month"),
          (f"rec_broken_share_{d}", "recurring", "all", d, f"rec_broken / rec_standing, {DIR_TXT[d]}"),
          (f"rec_broken_amt_share_{d}", "recurring", "all", d, f"share of standing dollars {DIR_TXT[d]} (last monthly amount) whose relationship broke"),
          (f"n_fi_{d}", "banks", "ext", d, f"distinct external banks of counterparties {DIR_TXT[d]} {WITH_TXT[d]}, all rails"),
          (f"n_fi_ach_{d}", "banks", "ext", d, f"distinct external banks {DIR_TXT[d]} {WITH_TXT[d]} on ACH only (attrition's fin_{d}_n definition)"),
          (f"top_fi_share_{d}", "banks", "ext", d, f"share of bank-identified external dollars {DIR_TXT[d]} through the largest bank"),
          (f"fi_coverage_{d}", "banks", "ext", d, f"share of external dollars {DIR_TXT[d]} whose counterparty bank is known"),
          (f"new_fi_{d}", "bank churn", "ext", d, f"banks {DIR_TXT[d]} {WITH_TXT[d]} this month, not seen in the previous {LOOKBACK} months"),
          (f"reactivated_fi_{d}", "bank churn", "ext", d, f"banks seen in the previous {LOOKBACK} months but not last month, back this month ({DIR_TXT[d]})"),
          (f"retained_fi_{d}", "bank churn", "ext", d, f"banks {DIR_TXT[d]} {WITH_TXT[d]} both last month and this month"),
          (f"lost_fi_{d}", "bank churn", "ext", d, f"banks {DIR_TXT[d]} {WITH_TXT[d]} last month but not this month"),
          (f"fi_retention_{d}", "bank churn", "ext", d, f"retained_fi / last month's banks, {DIR_TXT[d]}"),
          (f"fi_rec_broken_{d}", "bank churn", "ext", d, f"standing bank relationships ({REC_MIN} of {REC_WINDOW} months) absent this month, {DIR_TXT[d]}"),
          (f"fi_rec_broken_share_{d}", "bank churn", "ext", d, f"fi_rec_broken / standing bank relationships, {DIR_TXT[d]}"),
          (f"selfpay_amt_{d}", "self-pay", "ext", d, f"dollars {DIR_TXT[d]} {WITH_TXT[d]} counterparties with the customer's own name at other banks"),
          (f"selfpay_share_{d}", "self-pay", "ext", d, f"selfpay_amt_{d} / amt_ext_{d}"),
          (f"selfpay_n_fi_{d}", "self-pay", "ext", d, f"distinct other banks holding own-name accounts, {DIR_TXT[d]}"),
          (f"ext_business_share_{d}", "composition", "ext", d, f"share of external dollars {DIR_TXT[d]} with business-like counterparty names"),
          (f"int_tm_share_{d}", "composition", "int", d, f"share of internal dollars {DIR_TXT[d]} with TM customers"),
          (f"int_org_share_{d}", "composition", "int", d, f"share of internal dollars {DIR_TXT[d]} with organisations (party_type O)"),
          (f"n_naics2_{d}", "sector", "int", d, f"distinct NAICS2 sectors of internal counterparties (null below {MIN_NAICS_CP} counterparties with valid NAICS)"),
          (f"naics2_top_share_{d}", "sector", "int", d, "dollar share of the largest counterparty sector (same guard)"),
          (f"naics2_entropy_{d}", "sector", "int", d, "Shannon entropy (nats) of dollars across counterparty sectors (same guard)"),
          (f"same_naics2_share_{d}", "sector", "int", d, "share of sector-known internal dollars with counterparties in the customer's own NAICS2 (null if own NAICS unknown)")]
    M += [(f"share_{r}_{d}", "rails", "all", d, f"share of dollars {DIR_TXT[d]} on {r}") for r in RAILS]
    M.append((f"n_rails_{d}", "rails", "all", d, f"number of rails used, {DIR_TXT[d]}"))
M += [("net_flow_all", "flow", "all", "", "amt_all_in − amt_all_out"),
      ("net_flow_ext", "flow", "ext", "", "amt_ext_in − amt_ext_out"),
      ("passthrough_ratio", "flow", "all", "", "min(amt_all_in, amt_all_out) / max(...): 1 = everything received is paid out"),
      ("history_months", "churn", "", "", f"months of history behind the churn columns (< {LOOKBACK} before {LOOKBACK} months after {HISTORY_START})")]
METRIC_SPEC = M

_rows = [(c_, "identity", "", "", IDENTITY_DEF[c_]) for c_ in IDENTITY_COLS + ["month"]] + list(METRIC_SPEC)
write_full(spark.createDataFrame(_rows, "column string, family string, scope string, direction string, definition string"),
           "cust_metrics_dictionary")
print(f"{len(IDENTITY_COLS)} identity + {len(METRIC_SPEC)} metric columns; dictionary written to {T['cust_metrics_dictionary']}")

In [ ]:
EXPECTED_COLS = set(IDENTITY_COLS + [c_ for c_, *_ in METRIC_SPEC])
def metrics_current(m):
    """Built AND carrying every column of the current dictionary (months from an older notebook version are rebuilt)."""
    if not month_done("cust_metrics_monthly", m): return False
    return EXPECTED_COLS <= set(spark.read.parquet(f"{P['cust_metrics_monthly']}/month={m}").columns)

def reregister(key):
    """Drop + re-create the table definition from the newest month's schema and re-attach every month on disk."""
    fs = fs_for(P[key])
    months = sorted(s.getPath().getName().split("=", 1)[1] for s in fs.listStatus(JPath(P[key]))
                    if s.getPath().getName().startswith("month="))
    if not months: return
    schema = spark.read.parquet(f"{P[key]}/month={months[-1]}").schema
    spark.sql(f"DROP TABLE IF EXISTS {T[key]}")
    ensure_table(key, schema, True)
    for mm in months: add_partition(key, mm)
    spark.catalog.refreshTable(T[key])

MONTH_LOG = []
for m in RUN_MONTHS:
    if not FORCE and metrics_current(m):                          # state for m-1 is only needed for months still to build
        print(f"{m}: already built, skipped"); continue
    if month_done("cust_metrics_monthly", m) and not FORCE:
        print(f"{m}: built by an older notebook version, rebuilding")
    t0 = time.time()
    persp = perspective(m).persist(MD)
    with timed("perspective rows + pairs", m):
        build_pairs(m, persp)
    PR = spark.read.parquet(f"{P['cust_pair_month']}/month={m}")
    with timed("churn state + metrics", m):
        X, J, JF = metrics_for(m, persp, PR)
        write_month(X, "cust_metrics_monthly", m, "mdm_id", ["mdm_id"])
    for df_ in (persp, J, JF): df_.unpersist()
    prune_states(m)

    with timed("reconciliation", m):
        mt = spark.read.parquet(f"{P['cust_metrics_monthly']}/month={m}")
        got = mt.agg(*[F.sum(F.col(f"amt_int_{d}") + F.col(f"amt_ext_{d}") + F.col(f"amt_self_{d}")).alias(d) for d in D],
                     F.count("*").alias("customers"), F.sum(F.col("is_tm").cast("int")).alias("tm_customers"),
                     F.sum((~F.col("in_neo4j_customer")).cast("int")).alias("not_in_neo4j_customer"),
                     F.countDistinct("mdm_id").alias("distinct_mdm")).first()
        exp = EM.filter(F.col("month") == m).agg(
                    F.sum(F.when(F.col("src_kind") == "CUST", F.col("amount"))).alias("out"),
                    F.sum(F.when(F.col("dst_kind") == "CUST", F.col("amount"))).alias("in")).first()
        ok = (got["out"] == exp["out"]) and (got["in"] == exp["in"]) and (got["customers"] == got["distinct_mdm"])
    MONTH_LOG.append({"month": m, "customers": got["customers"], "tm_customers": got["tm_customers"],
                      "not_in_neo4j_customer": got["not_in_neo4j_customer"], "out_usd": float(got["out"] or 0),
                      "in_usd": float(got["in"] or 0), "ok": ok, "minutes": (time.time() - t0) / 60})
    with open("pkg_cust_metrics_runlog.csv", "a") as f:
        f.write(f"{dt.datetime.now().isoformat(timespec='seconds')},{m},{(time.time()-t0):.0f},{got['customers']},{ok}\n")
    print(f"{m}: done in {(time.time()-t0)/60:,.1f} min | reconciliation {'OK' if ok else 'MISMATCH ❌'}")
    if not ok:
        raise AssertionError(f"{m}: customer flows {got['out']}/{got['in']} vs edges {exp['out']}/{exp['in']}, "
                             f"rows {got['customers']} vs distinct {got['distinct_mdm']}")

In [ ]:
for k in ("cust_pair_month", "cust_metrics_monthly"):
    reregister(k)
print("table definitions refreshed from the current schema:", T["cust_pair_month"], T["cust_metrics_monthly"])

## §3 QA, storage, and a first look at the TM population

In [ ]:
if MONTH_LOG:
    show(pd.DataFrame(MONTH_LOG), "Per month: customers, reconciliation, time", money=("out_usd", "in_usd"))

built = [m for m in RUN_MONTHS if month_done("cust_metrics_monthly", m)]
st = []
for k in ("cust_pair_month", "cust_metrics_monthly"):
    tot = sum(size_of(f"{P[k]}/month={m}")[0] for m in built)
    st.append({"table": T[k], "months": len(built), "GB_total": tot / GB, "GB_per_month": tot / max(len(built), 1) / GB})
st.append({"table": "churn state (working, latest months only)", "months": KEEP_STATES,
           "GB_total": size_of(STATE_DIR)[0] / GB, "GB_per_month": size_of(STATE_DIR)[0] / GB / max(KEEP_STATES, 1)})
show(pd.DataFrame(st), "Storage")

if built:
    lm = built[-1]
    mt = spark.read.parquet(f"{P['cust_metrics_monthly']}/month={lm}").filter(F.col("is_tm"))
    look = ["ncpty_all_out", "ncpty_all_in", "n_fi_out", "n_fi_in", "n_fi_ach_out", "avg_ticket_out", "active_days_out",
            "share_ACH_out", "share_CHECK_out", "new_cpty_ext_out", "reactivated_cpty_ext_out", "lost_cpty_ext_out",
            "retention_out", "lost_amt_share_in", "rec_broken_share_out", "new_fi_in", "lost_fi_in", "fi_retention_in",
            "selfpay_share_out", "passthrough_ratio"]
    rows = []
    for col in look:
        r = mt.agg(F.expr(f"percentile_approx({col}, array(0.1, 0.5, 0.9))").alias("q"),
                   F.avg(F.col(col).isNull().cast("double")).alias("null_share"),
                   F.avg((F.col(col) > 0).cast("double")).alias("nonzero_share")).first()
        q = r["q"] or [None, None, None]
        rows.append({"metric": col, "p10": q[0], "p50": q[1], "p90": q[2], "null_share": r["null_share"], "nonzero_share": r["nonzero_share"]})
    show(pd.DataFrame(rows), f"TM customers, {lm}: distribution sanity check", pct=("null_share", "nonzero_share"))
    show(spark.table(T["cust_metrics_dictionary"]).toPandas(), "Data dictionary")

print("\n-- Impala: INVALIDATE METADATA once per new table, then REFRESH after every Spark run")
for k in ("cust_dim", "cust_pair_month", "cust_metrics_monthly", "cust_metrics_dictionary"):
    print(f"INVALIDATE METADATA {T[k]};")
for k in ("cust_dim", "cust_pair_month", "cust_metrics_monthly", "cust_metrics_dictionary"):
    print(f"REFRESH {T[k]};")
CD.unpersist()